In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax as dfx
import optax
import numpy as np
from jax import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = 'spirals.npz'
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 10000
USE_VALIDATION = True
VALIDATION_SPLIT = 0.2

INPUT_DIM = 3        # [x, y, time]
HIDDEN_DIM = 16
OUTPUT_DIM = 1

NUM_EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

OUTPUT_FILE = 'alpha_predictions.npy'

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(filepath, n_train=None, n_test=None):
    print("Loading data...")
    # Using 'allow_pickle=True' just in case, though standard saving doesn't need it
    data = np.load(filepath, allow_pickle=True)
    
    xy_train = data['xy_train'][:n_train] if n_train else data['xy_train']
    alpha_train = data['alpha_train'][:n_train] if n_train else data['alpha_train']
    xy_test = data['xy_test'][:n_test] if n_test else data['xy_test']
    
    seq_len = xy_train.shape[1]
    time = jnp.linspace(0, 1, seq_len)
    
    print("Adding time dimension...")
    def add_time(xy):
        # Optimized: Vectorized time addition
        # xy shape: (B, L, 2)
        # time shape: (L,) -> (1, L, 1)
        batch_size = xy.shape[0]
        time_expanded = jnp.tile(time[None, :, None], (batch_size, 1, 1))
        return jnp.concatenate([xy, time_expanded], axis=-1)
    
    train_data = add_time(jnp.array(xy_train))
    test_data = add_time(jnp.array(xy_test))
    
    print("Normalizing x, y coordinates...")
    xy_mean = train_data[:, :, :2].mean(axis=(0, 1))
    xy_std = train_data[:, :, :2].std(axis=(0, 1))
    
    # Safe normalization
    safe_std = xy_std + 1e-8
    train_data = train_data.at[:, :, :2].set((train_data[:, :, :2] - xy_mean) / safe_std)
    test_data = test_data.at[:, :, :2].set((test_data[:, :, :2] - xy_mean) / safe_std)
    
    alpha_train = jnp.array(alpha_train).squeeze()
    
    return train_data, alpha_train, test_data

# ============================================================================
# GRU-ODE MODEL (OPTIMIZED)
# ============================================================================

class ODEFunc(eqx.Module):
    mlp: eqx.nn.MLP
    hidden_size: int

    def __init__(self, hidden_size: int, *, key):
        self.hidden_size = hidden_size
        self.mlp = eqx.nn.MLP(
            in_size=hidden_size,
            out_size=hidden_size,
            width_size=hidden_size * 2,
            depth=1,
            activation=jax.nn.softplus,
            key=key,
        )

    def __call__(self, t, h, args):
        return self.mlp(h)


class GRUCell(eqx.Module):
    Wz: jnp.ndarray
    Wr: jnp.ndarray
    Wh: jnp.ndarray
    
    def __init__(self, input_size, hidden_size, key):
        key_z, key_r, key_h = random.split(key, 3)
        scale = 1.0 / jnp.sqrt(hidden_size)
        self.Wz = random.normal(key_z, (hidden_size + input_size, hidden_size)) * scale
        self.Wr = random.normal(key_r, (hidden_size + input_size, hidden_size)) * scale
        self.Wh = random.normal(key_h, (hidden_size + input_size, hidden_size)) * scale
    
    def __call__(self, x, h_prev):
        combined = jnp.concatenate([h_prev, x], axis=-1)
        z = jax.nn.sigmoid(combined @ self.Wz)
        r = jax.nn.sigmoid(combined @ self.Wr)
        combined_reset = jnp.concatenate([r * h_prev, x], axis=-1)
        h_prime = jnp.tanh(combined_reset @ self.Wh)
        h = (1 - z) * h_prime + z * h_prev
        return h


class GRUODERegressor(eqx.Module):
    ode_func: ODEFunc
    gru_cell: GRUCell
    regressor: eqx.nn.Linear
    hidden_size: int

    def __init__(self, input_size: int, hidden_size: int, output_size: int, *, key):
        key_ode, key_gru, key_reg = random.split(key, 3)
        self.hidden_size = hidden_size
        self.ode_func = ODEFunc(hidden_size, key=key_ode)
        self.gru_cell = GRUCell(input_size, hidden_size, key=key_gru)
        self.regressor = eqx.nn.Linear(hidden_size, output_size, key=key_reg)

    def __call__(self, trajectory):
        # trajectory: (seq_len, 3) [x, y, time]
        
        # Initial hidden state and time
        h0 = jnp.zeros((self.hidden_size,), dtype=jnp.float32)
        t0 = trajectory[0, 2] # Usually 0.0
        
        # Prepare solver once
        solver = dfx.Dopri5()
        term = dfx.ODETerm(self.ode_func)
        
        # --- OPTIMIZATION: Using jax.lax.scan instead of Python loop ---
        def scan_step(carry, inputs):
            h_prev, t_prev = carry
            obs_curr = inputs[:2] # x, y
            t_curr = inputs[2]    # time
            
            # 1. Evolve h_prev from t_prev to t_curr via ODE
            # We use a conditional to handle the very first step if t_prev == t_curr
            # or simply rely on diffeqsolve handling dt=0 gracefully.
            
            # Note: diffrax handles t0==t1 by returning y0 immediately, 
            # but setting a discrete dt0 can cause errors if t1-t0 is 0.
            dt = t_curr - t_prev
            target_dt0 = jnp.where(dt > 0, dt / 5.0, 0.01)

            solution = dfx.diffeqsolve(
                term, solver,
                t0=t_prev, t1=t_curr,
                dt0=target_dt0,
                y0=h_prev,
                max_steps=16, # Kept strictly low for speed as per original code
            )
            
            # The solution at t1 is the input to the GRU
            h_ode = solution.ys[0] # diffeqsolve returns structure matching y0
            
            # 2. Update with observation using GRU
            h_next = self.gru_cell(obs_curr, h_ode)
            
            return (h_next, t_curr), None

        # Run scan
        # We start the scan. Note: The first observation usually happens at t=0.
        # If t=0 for first step, the ODE step is skipped (dt=0) and just GRU update happens.
        (h_final, _), _ = jax.lax.scan(scan_step, (h0, t0), trajectory)
        
        return self.regressor(h_final).squeeze()

# ============================================================================
# TRAINING (OPTIMIZED)
# ============================================================================

def loss_fn(model, batch_trajectories, batch_alphas):
    """Batched MSE loss."""
    # Vectorize the model over the batch dimension
    # in_axes: (None, 0) -> model is fixed, trajectory is batched
    pred_fn = jax.vmap(model)
    preds = pred_fn(batch_trajectories)
    return jnp.mean((preds - batch_alphas) ** 2)

@eqx.filter_jit
def train_step(model, opt_state, batch_trajectories, batch_alphas, optimizer):
    # loss_fn now handles the batch internally via vmap
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, batch_trajectories, batch_alphas)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def evaluate_batch(model, data, alphas):
    """Efficient batched evaluation for validation."""
    pred_fn = jax.vmap(model)
    preds = pred_fn(data)
    mae = jnp.mean(jnp.abs(preds - alphas))
    rmse = jnp.sqrt(jnp.mean((preds - alphas) ** 2))
    return mae, rmse, preds

def train_model(model, train_data, train_alphas, val_data, val_alphas, 
                num_epochs, batch_size, key, use_validation=False):
    
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
    
    num_samples = train_data.shape[0]
    steps_per_epoch = num_samples // batch_size
    train_losses = []
    val_maes = []
    
    print(f"Starting training on device: {jax.devices()[0]}")
    
    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        key, subkey = random.split(key)
        perm = random.permutation(subkey, num_samples)
        
        # Shuffle data
        shuffled_data = train_data[perm]
        shuffled_alphas = train_alphas[perm]
        
        # Truncate to fit full batches for simplicity in JIT
        end_idx = steps_per_epoch * batch_size
        shuffled_data = shuffled_data[:end_idx]
        shuffled_alphas = shuffled_alphas[:end_idx]
        
        # Reshape to (Num_Batches, Batch_Size, ...) to scan over batches
        batched_data = shuffled_data.reshape(steps_per_epoch, batch_size, *train_data.shape[1:])
        batched_alphas = shuffled_alphas.reshape(steps_per_epoch, batch_size)
        
        epoch_loss = 0.0
        
        # Standard Python loop over batches is fine here because the heavy lifting 
        # (batch processing) is now vectorized inside train_step.
        # For extreme optimization, one could jax.lax.scan this loop too, 
        # but Python loop allows for tqdm progress bars.
        
        for i in range(steps_per_epoch):
            model, opt_state, loss = train_step(
                model, opt_state, batched_data[i], batched_alphas[i], optimizer
            )
            epoch_loss += loss.item() # .item() creates sync, but okay for loss reporting
            
        avg_loss = epoch_loss / steps_per_epoch
        train_losses.append(avg_loss)
        
        if use_validation:
            val_mae, val_rmse, _ = evaluate_batch(model, val_data, val_alphas)
            val_maes.append(val_mae.item())
            tqdm.write(f"Ep {epoch+1} | Loss: {avg_loss:.4f} | Val MAE: {val_mae:.4f}")
        else:
            tqdm.write(f"Ep {epoch+1} | Loss: {avg_loss:.4f}")
            
    return model, train_losses, val_maes

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Ensure JAX is using GPU/TPU if available
    # print(jax.devices())

    train_data, train_alphas, test_data = load_data(DATA_PATH, TRAIN_SAMPLES, TEST_SAMPLES)
    
    if USE_VALIDATION:
        n_train = int(train_data.shape[0] * (1 - VALIDATION_SPLIT))
        val_data = train_data[n_train:]
        val_alphas = train_alphas[n_train:]
        train_data = train_data[:n_train]
        train_alphas = train_alphas[:n_train]
    else:
        val_data = None
        val_alphas = None
    
    key = random.PRNGKey(RANDOM_SEED)
    key, model_key = random.split(key)
    
    model = GRUODERegressor(
        input_size=2,
        hidden_size=HIDDEN_DIM,
        output_size=OUTPUT_DIM,
        key=model_key
    )
    
    model, train_losses, val_maes = train_model(
        model, train_data, train_alphas, val_data, val_alphas,
        NUM_EPOCHS, BATCH_SIZE, key, USE_VALIDATION
    )
    
    print("\nTraining complete! Evaluating...")
    
    # Efficient Batched Evaluation
    # Note: If memory runs out on large test sets, chunk this using a loop
    # But for 10k samples, full vmap should fit on most GPUs.
    @eqx.filter_jit
    def predict_all(m, d):
        return jax.vmap(m)(d)

    # Process test set
    test_predictions = predict_all(model, test_data)
    
    # Save
    test_predictions_reshaped = np.array(test_predictions).reshape(-1, 1)
    np.save(OUTPUT_FILE, test_predictions_reshaped)
    print(f"Saved {test_predictions_reshaped.shape} predictions to {OUTPUT_FILE}")
    
    # --- Visualization Code (Kept mostly same, just ensuring array types) ---
    if USE_VALIDATION:
        train_mae, train_rmse, train_preds = evaluate_batch(model, train_data, train_alphas)
        val_mae, val_rmse, val_preds = evaluate_batch(model, val_data, val_alphas)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        axes[0, 0].plot(train_losses)
        axes[0, 0].set_title('Training Loss')
        
        axes[0, 1].plot(val_maes)
        axes[0, 1].set_title('Validation MAE')
        
        axes[1, 0].scatter(train_alphas, train_preds, alpha=0.5, s=5)
        axes[1, 0].set_title(f'Train (MAE: {train_mae:.4f})')
        axes[1, 0].grid(True)
        
        axes[1, 1].scatter(val_alphas, val_preds, alpha=0.5, s=5)
        axes[1, 1].set_title(f'Val (MAE: {val_mae:.4f})')
        axes[1, 1].grid(True)
        
        plt.tight_layout()
        plt.savefig('training_results.png')
        print("Plot saved.")

Loading data...
Adding time dimension...
Normalizing x, y coordinates...
Starting training on device: TFRT_CPU_0


Epochs:  20%|██        | 1/5 [00:45<03:00, 45.12s/it]

Ep 1 | Loss: 3.9699


Epochs:  40%|████      | 2/5 [01:28<02:11, 43.99s/it]

Ep 2 | Loss: 0.7698


...

Ep 3 | Loss: 0.7883


Epochs:  80%|████████  | 4/5 [02:55<00:43, 43.74s/it]

Ep 4 | Loss: 0.5628


Epochs: 100%|██████████| 5/5 [03:39<00:00, 43.97s/it]


Ep 5 | Loss: 0.4104

Training complete! Evaluating...
Saved (10000, 1) predictions to alpha_predictions.npy
